In [ ]:
dbutils.widgets.text("DBT_SELECT", "vault gold", "dbt --select targets (space-separated)")
dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog catalog")
dbutils.widgets.text("WAREHOUSE_ID", "53165753164ae80e", "SQL Warehouse ID")

In [ ]:
import subprocess, sys

print("Installing dbt-databricks...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "dbt-databricks==1.9.3"],
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(f"pip install failed:\n{r.stderr[-1000:]}")
print("dbt-databricks installed")

In [ ]:
import os, pathlib

dbt_select = dbutils.widgets.get("DBT_SELECT")
catalog     = dbutils.widgets.get("CATALOG")
warehouse_id = dbutils.widgets.get("WAREHOUSE_ID")

# Resolve host + token from Databricks runtime context
host  = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
http_path = f"/sql/1.0/warehouses/{warehouse_id}"

# Write a temporary profiles.yml
profiles_dir = pathlib.Path("/tmp/dbt_profiles")
profiles_dir.mkdir(exist_ok=True)
(profiles_dir / "profiles.yml").write_text(f"""cdc_gold:
  target: prod
  outputs:
    prod:
      type: databricks
      host: {host}
      http_path: {http_path}
      token: {token}
      catalog: {catalog}
      schema: gold
      threads: 4
""")
print(f"Profile written to {profiles_dir}/profiles.yml")
print(f"host={host}  http_path={http_path}")

In [ ]:
import subprocess, sys, pathlib, shutil, os

# Locate the dbt project directory (sibling of this notebook)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# Git-source jobs mount the repo at /Repos/.internal/<hash>/... (no /Workspace prefix).
# Try both forms so this notebook works in Repos AND in classic Workspace.
for prefix in ("", "/Workspace"):
    candidate = pathlib.Path(prefix + notebook_path).parent / "dbt_project"
    if candidate.exists():
        project_dir = candidate
        break
else:
    raise FileNotFoundError(f"Cannot find dbt_project relative to notebook path {notebook_path!r}")
print(f"dbt project dir: {project_dir}")

env = {**os.environ, "DBT_PROFILES_DIR": str(profiles_dir)}

# Find dbt binary — pip installs it into the venv's bin dir which may not be in PATH
dbt_bin = shutil.which("dbt")
if dbt_bin is None:
    # Search common Databricks serverless pip install locations
    for candidate in [
        pathlib.Path(sys.executable).parent / "dbt",
        pathlib.Path(sys.prefix) / "bin" / "dbt",
        pathlib.Path.home() / ".local" / "bin" / "dbt",
        pathlib.Path("/usr/local/bin/dbt"),
    ]:
        if candidate.exists():
            dbt_bin = str(candidate)
            break

if dbt_bin:
    print(f"dbt binary: {dbt_bin}")
else:
    # fallback: let subprocess find it (may fail with FileNotFoundError)
    dbt_bin = "dbt"
    print("WARNING: dbt not found via which/search, trying 'dbt' in PATH")

def run_dbt(args):
    cmd = [dbt_bin] + args
    print(f"\n$ {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(project_dir), env=env)
    if r.stdout: print(r.stdout[-3000:])
    if r.stderr: print(r.stderr[-1000:])
    if r.returncode != 0:
        out_tail = (r.stdout or "")[-2000:]
        err_tail = (r.stderr or "")[-1000:]
        raise RuntimeError(
            f"dbt command failed (exit {r.returncode}):\nSTDOUT:\n{out_tail}\nSTDERR:\n{err_tail}"
        )

# Skip dbt deps if packages are already committed in the repo
packages_dir = project_dir / "dbt_packages"
if not packages_dir.exists() or not any(packages_dir.iterdir()):
    print("dbt_packages not found — running dbt deps...")
    run_dbt(["deps"])
else:
    print(f"Using pre-committed packages in {packages_dir}")

for target in dbt_select.split():
    run_dbt(["build", "--select", target])

print("\ndbt build complete!")